<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-19</br>
</div>

</br>

In [3]:
# TODO 0: 실습을 위해 아래 패키지를 import 해주세요.
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")

사용 디바이스: cuda


</br>

# 학습 내용
>이번 장에서는 <strong>Linear Probing(선형 프로빙)</strong>에 대해 학습합니다.</br></br>
>사전학습 모델의 특성 추출 능력을 활용하여 분류 헤드만 학습하는 전이학습 기법을 학습해봅시다.</br></br>

</br>

# 전이학습 (Transfer Learning)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">사전학습된 모델의 지식을 새로운 태스크에 재활용</mark>하는 기법입니다.</br></br>
> 적은 데이터와 짧은 학습 시간으로 높은 성능을 달성할 수 있습니다.</br></br>

> 이미지 분류 모델을 처음부터 학습하려면 수백만 장의 레이블된 이미지와 수일에서 수주의 GPU 학습 시간이 필요합니다.</br> 대부분의 실무 상황에서는 이런 자원이 없습니다.</br></br>
> ImageNet(1,400만 장)으로 사전학습된 모델은 이미 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">에지, 질감, 형태, 객체 부분</mark> 등 다양한 시각 특징을 추출하는 방법을 알고 있으므로, 이 지식을 새로운 태스크에 재활용하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">소량의 데이터와 짧은 학습 시간</mark>으로 높은 성능을 달성할 수 있습니다.</br></br>
> 이러한 전이학습의 가장 간단한 형태가 **Linear Probing**입니다. 사전학습 모델의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">백본(Backbone) 전체를 동결(Freeze)</mark>하고, 마지막 분류 레이어(fc)만 교체하여 새로 학습합니다.</br></br>
> 학습 파라미터가 전체의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">0.05% 미만</mark>이므로 학습이 매우 빠르고, 데이터가 적을 때 과적합 위험이 낮으며, 사전학습 특징이 새 태스크에도 유효한지 빠르게 검증할 수 있습니다.</br></br>

In [1]:
# 백본 전체를 동결 -> 미분 동결 

</br>

## Linear Probing
> 사전학습 모델의 가중치를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">완전히 동결(Freeze)</mark>하고, 마지막 분류층(fc)만 교체하여 학습합니다.</br></br>

In [2]:
# 필터들이 여러 개 있음


In [5]:
# TODO 1: ResNet18 사전학습 모델을 model에 로드해봅시다. (weights=models.ResNet18_Weights.IMAGENET1K_V1)
import torchvision.models as models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

💡`weights` 파라미터란?
> `weights=ResNet18_Weights.IMAGENET1K_V1`은 ImageNet 데이터셋(130만 장, 1000개 클래스)으로 미리 학습된 가중치를 불러옵니다.</br></br>
> 이 가중치에는 엣지, 텍스처, 패턴 등 범용적인 시각 특징을 추출하는 능력이 이미 저장되어 있습니다.</br></br>
> `weights=None`으로 설정하면 랜덤 초기화된 모델이 로드되어 처음부터 학습해야 합니다.

In [10]:
# TODO 2: 모든 파라미터를 동결(requires_grad=False)해봅시다.
for param in model.parameters():
    param.requires_grad = False
    
# 동결 확인: 모든 파라미터의 requires_grad 상태 출력
for name, param in model.named_parameters():
    print(f"{name}: requires_grad = {param.requires_grad}")

conv1.weight: requires_grad = False
bn1.weight: requires_grad = False
bn1.bias: requires_grad = False
layer1.0.conv1.weight: requires_grad = False
layer1.0.bn1.weight: requires_grad = False
layer1.0.bn1.bias: requires_grad = False
layer1.0.conv2.weight: requires_grad = False
layer1.0.bn2.weight: requires_grad = False
layer1.0.bn2.bias: requires_grad = False
layer1.1.conv1.weight: requires_grad = False
layer1.1.bn1.weight: requires_grad = False
layer1.1.bn1.bias: requires_grad = False
layer1.1.conv2.weight: requires_grad = False
layer1.1.bn2.weight: requires_grad = False
layer1.1.bn2.bias: requires_grad = False
layer2.0.conv1.weight: requires_grad = False
layer2.0.bn1.weight: requires_grad = False
layer2.0.bn1.bias: requires_grad = False
layer2.0.conv2.weight: requires_grad = False
layer2.0.bn2.weight: requires_grad = False
layer2.0.bn2.bias: requires_grad = False
layer2.0.downsample.0.weight: requires_grad = False
layer2.0.downsample.1.weight: requires_grad = False
layer2.0.downsample.

💡`BatchNorm (bn)`이란?
> Conv 레이어의 출력값을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정규화(평균 0, 분산 1)</mark>하는 레이어입니다.</br></br>
> 레이어를 통과할수록 값의 분포가 불안정해지는 문제를 방지하여 학습을 안정적이고 빠르게 만들어줍니다.</br></br>
> `bn1`, `bn2`는 각각 첫 번째, 두 번째 Conv 뒤에 붙어 출력값을 정규화합니다.</br></br>
> Conv → BatchNorm → ReLU 순서가 CNN의 기본 패턴입니다.

In [13]:
# TODO 3: fc 레이어를 10개 클래스에 맞게 교체해봅시다.
model.fc = nn.Linear(model.fc.in_features, 10)
print(f"새 fc 레이어: {model.fc}")

# in_features=512 이니깐 512인걸 알 수 있음

새 fc 레이어: Linear(in_features=512, out_features=10, bias=True)


In [20]:
# TODO 4: 전체/학습 가능 파라미터 수를 출력해봅시다.
total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f"전체: {total_params:,}, 학습 가능: {trainable_params:,} ({trainable_params/total_params:.1%})")

전체: 11,181,642, 학습 가능: 5,130 (0.0%)


💡`numel()`이란?
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Number of Elements</mark>의 약자로, 텐서 안에 들어있는 숫자의 총 개수를 반환합니다.</br></br>
> 예: `(3, 4)` 크기의 텐서 → `numel() = 12`</br></br>
> 모델의 파라미터 수를 셀 때 `sum(p.numel() for p in model.parameters())`처럼 사용합니다.

💡왜 동결하는가?
> 사전학습된 특성 추출기는 이미 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">범용적인 시각 특성을 학습</mark>했습니다.</br>
> 적은 데이터로 전체를 학습하면 과적합 위험이 크므로, 헤드만 학습하는 것이 안전합니다.</br></br>

</br>

## 이미지 전처리 (transforms)
> `torchvision.transforms`는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이미지를 모델이 처리할 수 있는 형태로 변환</mark>하는 전처리 도구 모음입니다.</br></br>

>이미지는 크기, 비율, 색상 범위가 제각각이지만, 신경망은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">고정된 크기의 정규화된 텐서</mark>를 입력으로 받습니다.</br>
>`transforms`는 이 차이를 해결하는 **이미지 → 텐서 변환 파이프라인**입니다.</br></br>

>```
> 원본 이미지 (다양한 크기, 0~255)
>    ↓ Resize / CenterCrop  → 고정 크기 (224×224)
>    ↓ ToTensor()            → PIL 이미지 → PyTorch 텐서, [0,1] 범위
>    ↓ Normalize()           → 채널별 평균/표준편차로 정규화
>모델 입력 텐서 (C×H×W, 정규화됨)
>```

> 사전학습 모델은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 시 사용한 것과 동일한 전처리</mark>를 적용해야 합니다.</br>
> ResNet은 ImageNet 데이터셋(224×224, 특정 평균/표준편차)으로 학습되었으므로 같은 설정을 사용합니다.</br></br>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">Transform</th>
      <th style="text-align:center">역할</th>
      <th style="text-align:center">예시</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>Resize(256)</code></td><td style="text-align:center">이미지의 짧은 변을 256px로 조정</td><td style="text-align:center">다양한 크기 → 256px 통일</td></tr>
    <tr><td style="text-align:center"><code>CenterCrop(224)</code></td><td style="text-align:center">중앙에서 224×224 영역을 잘라냄</td><td style="text-align:center">ResNet 입력 크기에 맞춤</td></tr>
    <tr><td style="text-align:center"><code>ToTensor()</code></td><td style="text-align:center">PIL 이미지를 PyTorch 텐서로 변환</td><td style="text-align:center">[0,255] → [0.0, 1.0]</td></tr>
    <tr><td style="text-align:center"><code>Normalize(mean, std)</code></td><td style="text-align:center">채널별 평균/표준편차로 정규화</td><td style="text-align:center">ImageNet 통계값 사용</td></tr>
  </tbody>
</table>

> `transforms.Compose()`는 위 전처리를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">순서대로 연결</mark>하는 파이프라인입니다.</br></br>

In [23]:
# TODO 5: ImageNet 사전학습 모델에 맞는 전처리 파이프라인을 구성해봅시다.

# TODO 5-1: Resize(256)과 CenterCrop(224)을 적용해봅시다.
# TODO 5-2: ToTensor()로 텐서 변환해봅시다.
# TODO 5-3: ImageNet 평균/표준편차로 Normalize해봅시다.
#            ImageNet 128만 장의 RGB 채널별 평균/표준편차를 미리 계산한 고정값입니다.
#            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize(256),      # transform -> transforms 로 수정
    transforms.CenterCrop(224),  # transform -> transforms 로 수정
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    )
])

print(f"전처리 파이프라인: {transform}")

전처리 파이프라인: Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


💡Normalize 값
> ImageNet 사전학습 모델을 사용할 때는 반드시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ImageNet 통계값</mark>으로 정규화해야 합니다.</br>
> 다른 값을 사용하면 성능이 크게 하락합니다.</br></br>

</br>

## 데이터 준비 (CIFAR-10)
> CIFAR-10 데이터셋을 로드하고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ImageNet 사전학습 모델에 맞는 전처리</mark>를 적용합니다.</br>
> 위에서 정의한 `transform`을 사용하여 이미지를 변환합니다.</br></br>

In [49]:
# TODO 6: cifar10_train에 CIFAR-10 학습 데이터를 로드해봅시다. (transform 적용, download=True)
transform = transforms.Compose([
    transforms.ToTensor()
])

cifar10_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

print(f"학습 데이터: {len(cifar10_train)}개")
print(f"클래스: {cifar10_train.classes}")

학습 데이터: 50000개
클래스: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [50]:
# TODO 7: cifar10_test에 CIFAR-10 테스트 데이터를 로드해봅시다.
cifar10_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=True)

print(f"테스트 데이터: {len(cifar10_test)}개")

테스트 데이터: 10000개


In [51]:
# TODO 8: train_loader를 생성해봅시다. (batch_size=256, shuffle=True)

train_loader = DataLoader(cifar10_train, batch_size=256, shuffle=True)
print(f"train_loader: {len(train_loader)} 배치")

train_loader: 196 배치


In [52]:
# TODO 9: test_loader를 생성해봅시다. (batch_size=256, shuffle=False)
test_loader = DataLoader(cifar10_test, batch_size=256, shuffle=True)
print(f"test_loader: {len(test_loader)} 배치")

test_loader: 40 배치


</br>

## 모델 학습 (Training Loop)
> 손실 함수, 옵티마이저, 스케줄러를 정의하고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Linear Probing 방식으로 학습</mark>합니다.</br>
> `model.fc.parameters()`만 옵티마이저에 전달하여 분류 헤드만 학습합니다.</br></br>

In [31]:
# 위 사진을 에포크 이라고 한다.

In [53]:
# TODO 10: cross_entropy에 CrossEntropyLoss를 저장해봅시다.
cross_entropy = nn.CrossEntropyLoss()
print(f"손실 함수: {cross_entropy}")

손실 함수: CrossEntropyLoss()


In [54]:
# TODO 11: optimizer에 SGD를 저장해봅시다. (model.fc.parameters(), lr=0.001, momentum=0.9)
optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)
print(f"옵티마이저: {optimizer}")

옵티마이저: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0
)


💡`optim.SGD()`란?
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">확률적 경사 하강법(Stochastic Gradient Descent)</mark> 옵티마이저입니다.</br></br>
> `lr` (학습률): 파라미터를 한 번에 얼마나 조정할지 결정합니다. 너무 크면 발산, 너무 작으면 학습이 느립니다.</br></br>
> `momentum`: 이전 업데이트 방향을 기억하여 같은 방향이면 가속합니다. 보통 0.9를 사용합니다.</br></br>

In [55]:
# 파라미터가 크면 튕겨서 위로 올라감

In [56]:
# TODO 12: scheduler에 StepLR을 저장해봅시다. (step_size=2, gamma=0.1)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)
print(f"스케줄러: {scheduler}")

스케줄러: <torch.optim.lr_scheduler.StepLR object at 0x728e25438b90>


💡`StepLR` 스케줄러란?
> 학습이 진행될수록 학습률을 점차 줄여주는 도구입니다.</br></br>
> `step_size=2`: 2 에폭마다 학습률을 줄입니다.</br></br>
> `gamma=0.1`: 줄일 때 현재 학습률의 10%로 감소시킵니다.</br></br>
> 처음에는 크게 학습하고, 나중에는 세밀하게 조정하는 효과가 있습니다.</br></br>

In [69]:
# TODO 13: 학습 루프를 구현해봅시다.
model = model.to(device)
epochs = 5
for epoch in range(epochs):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        # TODO 13-1: 기울기를 초기화해봅시다.
        optimizer.zero_grad() #기울기가 누적되므로 초기화 시킨다.
        
        # TODO 13-2: 순전파를 수행하고 outputs에 저장해봅시다.
        outputs = model(images) #예측
        
        # TODO 13-3: 손실을 계산해봅시다.
        loss = cross_entropy(outputs, labels)
        
        # TODO 13-4: 역전파를 수행해봅시다.
        loss.backward()
    
        # TODO 13-5: 파라미터를 업데이트해봅시다.
        optimizer.step()

    # TODO 13-6: 에폭 종료 후 스케줄러를 업데이트해봅시다.
    scheduler.step()
    # print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}, Train Acc: {epoch_accuracy:.2f}%")

</br>

## 모델 평가 (Evaluation)
> 학습된 모델을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">테스트 데이터셋으로 평가</mark>하여 Linear Probing의 성능을 확인합니다.</br></br>

In [64]:
# TODO 14: 모델을 평가 모드로 전환해봅시다.
model.eval()
print(f"모델 모드: training={model.training}")

모델 모드: training=False


In [70]:
# TODO 15: torch.no_grad() 블록 안에서 테스트 데이터를 순회하며 정확도를 계산해봅시다.
total_count = 0
correct_count = 0

with torch.no_grad():
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, pred = outputs.max(1)
        total_count += labels.size(0)
        correct_count += pred.eq(labels).sum().item()

test_accuracy = 100.0 * correct_count / total_count
print(f"Linear Probing 테스트 정확도: {test_accuracy:.2f}%")

Linear Probing 테스트 정확도: 43.05%


💡model.eval()의 역할
> `model.eval()`은 모델을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">평가 모드</mark>로 전환합니다.</br>
> Dropout이 비활성화되고, BatchNorm은 학습 중 계산된 running statistics를 사용합니다.</br>
> 추론 시에는 반드시 `torch.no_grad()`와 함께 사용하여 불필요한 그래디언트 계산을 방지합니다.</br></br>